In [1]:
import importlib.util
import subprocess
import sys

def check_and_install(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    
    # Verificar si el paquete está instalado
    spec = importlib.util.find_spec(import_name)
    if spec is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"✅ {package_name} instalado correctamente.")
    else:
        print(f"📦 {package_name} ya está presente.")

# Ejecutar comprobación e instalación
check_and_install("ultralytics")
check_and_install("opencv-python", "cv2")

# Mostrar versiones finales
import ultralytics
import cv2
print("-" * 30)
print(f"🚀 Versión de Ultralytics: {ultralytics.__version__}")
print(f"📷 Versión de OpenCV: {cv2.__version__}")
print("-" * 30)

# Verificación de hardware para YOLO
ultralytics.checks()

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.9/112.6 GB disk)


In [2]:
from google.colab import drive
import os

# Conectar a Drive
drive.mount('/content/drive')

# Definir la ruta de la carpeta del proyecto
path_ppe = '/content/drive/MyDrive/ppe'

# Crear la carpeta 'ppe' si no existe
if not os.path.exists(path_ppe):
    os.makedirs(path_ppe)
    print(f"✅ Carpeta creada en: {path_ppe}")
else:
    print(f"📦 La carpeta '{path_ppe}' ya existe.")

Mounted at /content/drive
✅ Carpeta creada en: /content/drive/MyDrive/ppe


In [3]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 137.4 MB/s eta 0:00:0000:01


In [4]:
from roboflow import Roboflow

# Configuración de Roboflow
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")

# Descargar el dataset en formato YOLOv8
# Nota: Cambia el número de versión (1, 2, etc.) según la que quieras bajar
dataset = project.version(1).download("yolov8")

# Ruta del archivo data.yaml para usar en el entrenamiento
print(f"Dataset descargado en: {dataset.location}")
path_yaml = f"{dataset.location}/data.yaml"

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ppe-factory-1 in yolov8:: 100%|██████████| 21608/21608 [00:04<00:00, 4961.54it/s]

Dataset descargado en: /content/ppe-factory-1


In [5]:
!ls

drive  ppe-factory-1


In [6]:
import os
import shutil
import yaml
from google.colab import drive
from roboflow import Roboflow
from ultralytics import YOLO

# 1. Montar Drive y preparar carpetas
drive.mount('/content/drive')
path_ppe = '/content/drive/MyDrive/ppe'
dataset_dest = os.path.join(path_ppe, "dataset_ppe")

if not os.path.exists(path_ppe):
    os.makedirs(path_ppe)

# 2. Descargar de Roboflow
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")
dataset = project.version(1).download("yolov8")

# 3. Mover dataset a Drive (si no existe ya)
if not os.path.exists(dataset_dest):
    shutil.move(dataset.location, dataset_dest)
    print(f"✅ Dataset movido a Drive: {dataset_dest}")
else:
    print("📦 El dataset ya existía en Drive.")

# 4. Corregir rutas en el archivo data.yaml (CRUCIAL para que YOLO no falle)
path_yaml = os.path.join(dataset_dest, "data.yaml")
with open(path_yaml, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = os.path.join(dataset_dest, "train/images")
config['val'] = os.path.join(dataset_dest, "valid/images")
if 'test' in config:
    config['test'] = os.path.join(dataset_dest, "test/images")

with open(path_yaml, 'w') as f:
    yaml.dump(config, f)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
loading Roboflow workspace...
loading Roboflow project...
✅ Dataset movido a Drive: /content/drive/MyDrive/ppe/dataset_ppe


In [ ]:
# 5. Cargar Modelo YOLO8 e INICIAR ENTRENAMIENTO
model = YOLO("yolov8n.pt")

model.train(
    data=path_yaml,
    epochs=100,
    imgsz=640,
    project=path_ppe,        # Guarda resultados en Drive
    name="entrenamiento_ppe" # Nombre de la carpeta de resultados
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe/dataset_ppe/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=entrenamiento_ppe, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 